In [ ]:
from typing_extensions import TypedDict  # for defining the structure of the state
from langgraph.graph import StateGraph, START, END  # import necessary components from langgraph, START è il nodo iniziale speciale, END è il nodo finale speciale

In [ ]:
class SimpleState(TypedDict):
    input_text: str
    output_text: str

SimpleState(input_text="hello", output_text="HELLO")

In [ ]:
def process_input(state: SimpleState) -> SimpleState:
    output_text = state['input_text'].upper() #'hello'->'HELLO'
    return {'output_text': output_text}  #RESTITUISCI L'AGGIORNAMENTO (non necessario return tutttto lo stato (cioe input_text & output_text))

state = {'input_text': 'hello', 'output_text': ''}  #crei stato iniziale!
process_input(state)  #chiami manualmente il nodo (cioe la funzione)

In [ ]:
def add_prefix(state:SimpleState):  #un'altro nodo
    print("Current State [Prefix]", state)  #debug print, print e.g. 'Current State [Prefix] { "output_text": "HELLO" } '
    output = "Hey, i have added something here. " + state['output_text']
    return {'output_text': output}  #update state!

add_prefix(process_input(state))  
#state final so will be {'output_text': 'Hey, i have added something here. HELLO'}

In [ ]:
def add_suffix(state:SimpleState):
    print("Current State [Suffix]", state)
    output = state["output_text"] + ". i have added suffix!"
    return {'output_text': output}
add_suffix(add_prefix(process_input(state)))

In [ ]:

def create_simple_graph():
    builder = StateGraph(SimpleState)

    # add the nodes
    builder.add_node("process_input", process_input)
    builder.add_node("add_prefix", add_prefix)
    builder.add_node("add_suffix", add_suffix)

    #create connections adding the edges
    builder.add_edge(START, "process_input")
    builder.add_edge("process_input", "add_prefix")
    builder.add_edge("add_prefix", "add_suffix")
    builder.add_edge("add_suffix", END)

    # make it langchain runnable
    graph = builder.compile()
    return graph

graph = create_simple_graph()  #assign the graph to a variable so you can invoke it later
graph  

In [ ]:
# initial_state or user input

initial_state= {'input_text': "hello"}
graph.invoke(initial_state)

initial_state= {'input_text': "hello", "some_other_state": 'hi'}
graph.invoke(initial_state)

initial_state= {'input_text': "hello", "output_text": 'hi'}
graph.invoke(initial_state)